# Gold | Dataset de features para a Fase 3 (ML)
`gold.ds_features_fase3`: uma linha por município (rede municipal) e ano **T**. O alvo (`alvo_atingiu_meta_t`) vem de T;
**todas as features vêm só de T-1 ou antes** (sem vazamento do alvo). Anos sem T-1 ficam fora do dataset.
Alvo é 1/0 apenas quando a situação é ATINGIU_OU_SUPEROU / ABAIXO_DA_META; demais estados ficam de fora (não viram 0).

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
_m = iniciar_etapa()
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold_staging")
spark.sql("""
CREATE OR REPLACE TABLE workspace.gold_staging.ds_features_fase3 USING DELTA AS
SELECT t.ano, t.id_municipio, t.serie, t.rede_codigo,
       CASE t.situacao_meta WHEN 'ATINGIU_OU_SUPEROU' THEN 1 WHEN 'ABAIXO_DA_META' THEN 0 END AS alvo_atingiu_meta_t,
       p.taxa_alfabetizacao        AS taxa_alfabetizacao_t1,
       p.meta_alfabetizacao        AS meta_t1,
       p.diferenca_meta            AS diferenca_meta_t1,
       r.media_portugues           AS media_portugues_t1,
       r.proporcao_aluno_nivel_0 + r.proporcao_aluno_nivel_1 AS proporcao_niveis_baixos_t1,
       d.sigla_uf, d.regiao
FROM workspace.gold_staging.ft_meta_vs_resultado_municipio t
JOIN workspace.gold_staging.ft_meta_vs_resultado_municipio p
  ON p.id_municipio = t.id_municipio AND p.serie = t.serie AND p.rede_codigo = t.rede_codigo AND p.ano = t.ano - 1
JOIN workspace.gold_staging.ft_resultado_municipio r
  ON r.id_municipio = p.id_municipio AND r.serie = p.serie AND r.rede_codigo = p.rede_codigo AND r.ano = p.ano
LEFT JOIN workspace.gold_staging.dim_municipio d ON d.id_municipio = t.id_municipio
WHERE t.situacao_meta IN ('ATINGIU_OU_SUPEROU', 'ABAIXO_DA_META')
""")
registrar_execucao("GOLD", "ds_features_fase3", _m, row_count=contar_tabela("workspace.gold_staging.ds_features_fase3"))